In [ ]:
#
# ⚡ UNIVERSAL FIRST CELL - Run this FIRST in every notebook!
# Compatible with: Google Colab, GitHub Codespaces, Local
#

import os
import subprocess
import sys

# Detect environment
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)

print(f"\U0001f680 Environment: {'Colab' if IS_COLAB else 'Codespaces' if IS_CODESPACES else 'Local'}")

# Ensure we are in the root directory for relative paths
while not os.path.exists('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print(f"\U0001f4c1 Working directory set to: {os.getcwd()}")

# Install gridstatus on Colab/Codespaces
if not IS_LOCAL:
    !pip install gridstatus -q

print("\u2705 Environment ready!")

# 14 - Gridstatus ISO Queue Cross-Reference & Labeling

**Goal**: Pull ISO interconnection queue data for all 7 major US ISOs via gridstatus, cross-reference with buildout event announcements, and label each event as `kept`, `failed`, `pending`, or `unmatched`.

**Pipeline**:
1. Load buildout events from `data/raw/buildout_events_raw.csv`
2. Pull interconnection queue data from CAISO, MISO, PJM, ERCOT, NYISO, SPP, ISONE
3. Merge all ISO queue data into unified DataFrame
4. Match events to queue entries by: company name, location, MW range (±20%), date proximity
5. Label: Operating=kept, Withdrawn/Suspended=failed, In Queue=pending, no match=unmatched
6. Save labeled output to `data/processed/buildout_promises_real.csv`

**Output columns**: url, source_domain, announcement_date, company, location_city, location_state, mw_capacity, target_completion_date, is_buildout, confidence, iso_region, queue_status, queue_date, queue_mw, queue_county, actual_completion_date, label

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import warnings
from datetime import datetime, timedelta
from pathlib import Path
warnings.filterwarnings('ignore')

# Gridstatus
try:
    import gridstatus
    HAS_GRIDSTATUS = True
    print("\u2705 gridstatus available")
except ImportError:
    HAS_GRIDSTATUS = False
    print("\u26a0\ufe0f gridstatus not installed (pip install gridstatus)")
    raise SystemExit(0)

print("\u2705 Libraries imported")

## Step 1: Load Buildout Events

Load the structured buildout event data produced by `13-article-extraction.ipynb`.

In [ ]:
# Load input data
INPUT_PATH = 'data/raw/buildout_events_raw.csv'

if not os.path.exists(INPUT_PATH):
    print(f"\u274c Input file not found: {INPUT_PATH}")
    print("   Run 13-article-extraction.ipynb first to generate buildout events.")
    raise SystemExit(0)

df_events = pd.read_csv(INPUT_PATH)
print(f"\U0001f4c2 Loaded {len(df_events)} buildout events from {INPUT_PATH}")
print(f"   Columns: {list(df_events.columns)}")

# Filter to only buildout events
if 'is_buildout' in df_events.columns:
    n_total = len(df_events)
    df_events = df_events[df_events['is_buildout'] == True].copy()
    print(f"   Filtered to {len(df_events)} buildout events (from {n_total} total)")

# Show summary by company
if 'company' in df_events.columns:
    print(f"\n\U0001f3f7\ufe0f Events by company:")
    print(df_events['company'].fillna('unknown').value_counts().to_string())

# Show summary by state
if 'location_state' in df_events.columns:
    state_col = df_events['location_state'].fillna('unknown')
    print(f"\n\U0001f4cd Events by state:")
    print(state_col.value_counts().to_string())

# Show MW range
if 'mw_capacity' in df_events.columns:
    has_mw = df_events['mw_capacity'].notna()
    print(f"\n\u26a1 Events with MW capacity: {has_mw.sum()} / {len(df_events)}")
    if has_mw.sum() > 0:
        print(f"   MW range: {df_events.loc[has_mw, 'mw_capacity'].min():.0f} - {df_events.loc[has_mw, 'mw_capacity'].max():.0f}")

## Step 2: Define ISO & Company-to-State Mapping

Map states to their likely ISO region for company-to-ISO assignment and better matching.

In [ ]:
# ISO definitions with their classes and state coverage
ISO_DEFS = [
    ('CAISO', gridstatus.CAISO, 'CAISO', [
        'CA',
    ]),
    ('MISO', gridstatus.MISO, 'MISO', [
        'IL', 'IN', 'MI', 'MN', 'WI', 'IA', 'MO', 'AR', 'LA', 'MS', 'ND', 'SD',
    ]),
    ('PJM', gridstatus.PJM, 'PJM', [
        'PA', 'NJ', 'MD', 'DE', 'DC', 'OH', 'WV', 'VA', 'NC', 'KY', 'TN',
    ]),
    ('ERCOT', gridstatus.ERCOT, 'ERCOT', [
        'TX',
    ]),
    ('NYISO', gridstatus.NYISO, 'NYISO', [
        'NY',
    ]),
    ('SPP', gridstatus.SPP, 'SPP', [
        'KS', 'NE', 'OK', 'NM',
    ]),
    ('ISONE', gridstatus.ISONE, 'ISONE', [
        'CT', 'MA', 'ME', 'NH', 'RI', 'VT',
    ]),
]

# Build reverse mapping: state -> ISO
STATE_TO_ISO = {}
for iso_name, iso_class, iso_code, states in ISO_DEFS:
    for state in states:
        STATE_TO_ISO[state] = iso_code
    # Also add states like TX panhandle -> ERCOT/SPP under SPP

# SPP also covers parts of AR, LA, MO, SD, ND, MN, IA, TX
for state in ['AR', 'LA', 'MO', 'SD', 'ND', 'MN', 'IA', 'TX']:
    if state not in STATE_TO_ISO:
        STATE_TO_ISO[state] = 'SPP'
    else:
        # These states are split between ISOs; mark as primary
        pass

print(f"\u2705 Mapped {len(STATE_TO_ISO)} states/territories to ISOs")
print(f"\U0001f310 ISO coverage:")
for iso_name, _, iso_code, states in ISO_DEFS:
    print(f"   {iso_code}: {', '.join(states)}")

## Step 3: Pull ISO Interconnection Queue Data

Try each ISO individually. One failure doesn't block others.

In [ ]:
# Pull queue data from each ISO
queue_dfs = []
failed_isos = []

for iso_name, iso_class, iso_code, states in ISO_DEFS:
    print(f"\n\U0001f3ed Pulling {iso_name} ({iso_code}) queue data...", end=' ')
    try:
        iso = iso_class()
        qdf = iso.get_interconnection_queues()
        if qdf is not None and len(qdf) > 0:
            # Add ISO region column
            qdf['iso_region'] = iso_code
            queue_dfs.append(qdf)
            print(f"\u2705 {len(qdf)} projects loaded")
        else:
            print(f"\u26a0\ufe0f Empty result (0 projects)")
    except Exception as e:
        failed_isos.append(iso_code)
        print(f"\u274c FAILED: {e}")

print(f"\n\n{'='*60}")
print(f"\U0001f4ca Queue data summary:")
print(f"   Successful ISOs: {len(queue_dfs)} / {len(ISO_DEFS)}")
if failed_isos:
    print(f"   Failed ISOs: {', '.join(failed_isos)}")
total_queue_projects = sum(len(qdf) for qdf in queue_dfs) if queue_dfs else 0
print(f"   Total queue projects: {total_queue_projects}")

In [ ]:
# Merge all ISO queue data into unified DataFrame
if not queue_dfs:
    print("\u274c No queue data loaded from any ISO. Cannot proceed.")
    raise SystemExit(0)

df_queue = pd.concat(queue_dfs, ignore_index=True)
print(f"\U0001f4ca Unified queue DataFrame: {len(df_queue)} projects")
print(f"   Columns: {list(df_queue.columns)}")

# Show project count by ISO
print(f"\n--- By ISO ---")
print(df_queue['iso_region'].value_counts().to_string())

# Show status breakdown
status_col = None
for col in ['Status', 'status', 'queue_status', 'QUEUE_STATUS']:
    if col in df_queue.columns:
        status_col = col
        break
if status_col:
    print(f"\n--- By Queue Status ---")
    print(df_queue[status_col].value_counts().to_string())

# Show MW range
mw_col = None
for col in ['Capacity (MW)', 'capacity_mw', 'Capacity MW', 'MW', 'capacity']:
    if col in df_queue.columns:
        mw_col = col
        break
if mw_col:
    print(f"\n--- MW Range ---")
    valid_mw = df_queue[mw_col].notna()
    if valid_mw.sum() > 0:
        print(f"   Range: {df_queue.loc[valid_mw, mw_col].min():.0f} - {df_queue.loc[valid_mw, mw_col].max():.0f} MW")
        print(f"   Median: {df_queue.loc[valid_mw, mw_col].median():.0f} MW")
        print(f"   Total: {df_queue.loc[valid_mw, mw_col].sum():,.0f} MW")

## Step 4: Normalize Queue Field Names

Gridstatus returns different field names across ISOs. Normalize to a consistent schema.

In [ ]:
# Detect and normalize queue column names
col_map = {}

# Status column
for candidate in ['Status', 'status', 'queue_status', 'QUEUE_STATUS', 'project_status']:
    if candidate in df_queue.columns:
        col_map['queue_status'] = candidate
        break

# MW / Capacity column
for candidate in ['Capacity (MW)', 'capacity_mw', 'Capacity MW', 'MW', 'capacity', 'Capacity', 'Size (MW)']:
    if candidate in df_queue.columns:
        col_map['queue_mw'] = candidate
        break

# Queue Date column
for candidate in ['Queue Date', 'queue_date', 'Queue Date (Application)', 'application_date', 'queue_date_submitted']:
    if candidate in df_queue.columns:
        col_map['queue_date'] = candidate
        break

# County / Location column
for candidate in ['County', 'county', 'County (State)', 'location', 'Location', 'State', 'state']:
    if candidate in df_queue.columns:
        col_map['queue_county'] = candidate
        break

# Project Name column
for candidate in ['Project Name', 'project_name', 'Project', 'project', 'Queue Name']:
    if candidate in df_queue.columns:
        col_map['project_name'] = candidate
        break

# Withdrawn Date column
for candidate in ['Withdrawn Date', 'withdrawn_date', 'Withdrawal Date', 'Date Withdrawn']:
    if candidate in df_queue.columns:
        col_map['withdrawn_date'] = candidate
        break

# Actual Completion Date column
for candidate in ['Actual Completion Date', 'actual_completion_date', 'Completion Date', 'Commercial Operation Date',
                  'COD', 'In Service Date', 'Operating Date']:
    if candidate in df_queue.columns:
        col_map['actual_completion_date'] = candidate
        break

print(f"\U0001f4cb Normalized queue columns:")
for normalized, original in col_map.items():
    print(f"   {normalized} <- {original}")

# Rename to normalized names
rename_dict = {v: k for k, v in col_map.items()}
df_queue = df_queue.rename(columns=rename_dict)

# Ensure standard columns exist
for col in ['queue_status', 'queue_mw', 'queue_date', 'queue_county', 'project_name']:
    if col not in df_queue.columns:
        df_queue[col] = None
        print(f"   + Created missing column: {col} (filled with None)")

print(f"\n\u2705 Normalized queue shape: {df_queue.shape}")

## Step 5: Normalize Event Field Names

Ensure consistent field names in events DataFrame.

In [ ]:
# Detect event columns
event_col_map = {}

# Date columns
for candidate in ['announcement_date', 'date', 'DATE', 'published_date', 'ArticleDate']:
    if candidate in df_events.columns:
        event_col_map['announcement_date'] = candidate
        break
if 'announcement_date' not in event_col_map:
    event_col_map['announcement_date'] = 'announcement_date'
    if 'announcement_date' not in df_events.columns:
        df_events['announcement_date'] = None

# Company column
for candidate in ['company', 'Company', 'company_name', 'matched_company', 'v2_organizations']:
    if candidate in df_events.columns:
        event_col_map['company'] = candidate
        break
if 'company' not in event_col_map:
    event_col_map['company'] = 'company'
    if 'company' not in df_events.columns:
        df_events['company'] = None

# MW capacity
for candidate in ['mw_capacity', 'MW', 'mw', 'capacity_mw', 'Capacity_MW', 'promised_mw']:
    if candidate in df_events.columns:
        event_col_map['mw_capacity'] = candidate
        break
if 'mw_capacity' not in event_col_map:
    event_col_map['mw_capacity'] = 'mw_capacity'
    if 'mw_capacity' not in df_events.columns:
        df_events['mw_capacity'] = None

# City
for candidate in ['location_city', 'city', 'City', 'location']:
    if candidate in df_events.columns:
        event_col_map['location_city'] = candidate
        break
if 'location_city' not in event_col_map:
    event_col_map['location_city'] = 'location_city'
    if 'location_city' not in df_events.columns:
        df_events['location_city'] = None

# State
for candidate in ['location_state', 'state', 'State', 'location_state_code']:
    if candidate in df_events.columns:
        event_col_map['location_state'] = candidate
        break
if 'location_state' not in event_col_map:
    event_col_map['location_state'] = 'location_state'
    if 'location_state' not in df_events.columns:
        df_events['location_state'] = None

# Target completion date
for candidate in ['target_completion_date', 'target_date', 'completion_date', 'Target Completion Date']:
    if candidate in df_events.columns:
        event_col_map['target_completion_date'] = candidate
        break
if 'target_completion_date' not in event_col_map:
    event_col_map['target_completion_date'] = 'target_completion_date'
    if 'target_completion_date' not in df_events.columns:
        df_events['target_completion_date'] = None

print(f"\U0001f4cb Normalized event columns:")
for normalized, original in event_col_map.items():
    print(f"   {normalized} <- {original}")
print(f"\n\u2705 Events shape: {df_events.shape}")

## Step 6: Parse Dates

Convert date columns to datetime for proximity matching.

In [ ]:
def parse_dates_robust(series):
    """Try multiple date formats."""
    result = pd.to_datetime(series, errors='coerce', infer_datetime_format=True)
    return result

# Parse event dates
for col in ['announcement_date', 'target_completion_date']:
    if col in df_events.columns:
        parsed = parse_dates_robust(df_events[col])
        n_parsed = parsed.notna().sum()
        df_events[col] = parsed
        print(f"   Event '{col}': {n_parsed}/{len(df_events)} dates parsed")

# Parse queue dates
for col in ['queue_date', 'withdrawn_date', 'actual_completion_date']:
    if col in df_queue.columns and df_queue[col] is not None:
        parsed = parse_dates_robust(df_queue[col])
        n_parsed = parsed.notna().sum()
        df_queue[col] = parsed
        print(f"   Queue '{col}': {n_parsed}/{len(df_queue)} dates parsed")

# Parse queue MW as numeric
if 'queue_mw' in df_queue.columns:
    df_queue['queue_mw'] = pd.to_numeric(df_queue['queue_mw'], errors='coerce')
    print(f"   Queue 'queue_mw': {df_queue['queue_mw'].notna().sum()}/{len(df_queue)} parsed as numeric")

# Parse event MW as numeric
if 'mw_capacity' in df_events.columns:
    df_events['mw_capacity'] = pd.to_numeric(df_events['mw_capacity'], errors='coerce')
    print(f"   Event 'mw_capacity': {df_events['mw_capacity'].notna().sum()}/{len(df_events)} parsed as numeric")

## Step 7: Assign ISO Region to Each Event

Assign each event to the most likely ISO based on state location.

In [ ]:
# Assign ISO region based on state
def assign_iso(row):
    state = row['location_state']
    if pd.isna(state):
        return None
    state = str(state).upper().strip()
    return STATE_TO_ISO.get(state, None)

df_events['iso_region'] = df_events.apply(assign_iso, axis=1)

assigned = df_events['iso_region'].notna().sum()
print(f"\U0001f4cd ISO region assigned for {assigned}/{len(df_events)} events")
print(f"\n--- ISO Distribution ---")
print(df_events['iso_region'].value_counts(dropna=False).to_string())

# For events with no state, try to infer from company or city
no_iso = df_events[df_events['iso_region'].isna()]
if len(no_iso) > 0:
    print(f"\n\u26a0\ufe0f {len(no_iso)} events with unassigned ISO:")
    for _, row in no_iso.iterrows():
        city = row.get('location_city', '?')
        company = row.get('company', '?')
        print(f"   Company={company}, City={city}")

## Step 8: Define Matching Functions

Multi-step cross-reference: company name (fuzzy), location (state + county/city), MW range (±20%), date proximity (6 months).

In [ ]:
# Normalize company name for matching
def normalize_company(name):
    """Normalize company name for comparison."""
    if pd.isna(name):
        return ''
    name = str(name).lower().strip()
    # Remove common suffixes
    name = re.sub(r'\b(inc|llc|ltd|corp|corporation|company|technologies|technology|group|holdings|solutions|services|systems|na|north america|usa)\b', '', name)
    # Remove punctuation and extra whitespace
    name = re.sub(r'[^a-z0-9\s]', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# Normalize location (city/county) for matching
def normalize_location(loc):
    """Normalize location string for comparison."""
    if pd.isna(loc):
        return ''
    loc = str(loc).lower().strip()
    # Remove county/city suffixes
    loc = re.sub(r'\b(county|city|town|township|parish|cdp)\b', '', loc)
    loc = re.sub(r'[^a-z0-9\s]', '', loc)
    loc = re.sub(r'\s+', ' ', loc).strip()
    return loc

def company_name_match(event_company, queue_name):
    """Check if company names match.
    Returns a score 0-1 where 1=exact, 0.5=partial, 0=no match."""
    ec = normalize_company(event_company)
    qn = normalize_company(queue_name)
    if not ec or not qn:
        return 0.0
    # Exact match after normalization
    if ec == qn:
        return 1.0
    # One contains the other
    if ec in qn or qn in ec:
        return 0.8
    # Check word overlap
    ec_words = set(ec.split())
    qn_words = set(qn.split())
    if len(ec_words) > 0 and len(qn_words) > 0:
        overlap = len(ec_words & qn_words) / max(len(ec_words), len(qn_words))
        if overlap >= 0.5:
            return 0.6 * overlap
    return 0.0


def location_match(event_city, event_state, queue_county):
    """Check if locations match.
    Returns score 0-1."""
    if pd.isna(event_state) or pd.isna(queue_county):
        return 0.0
    
    q_loc = normalize_location(str(queue_county))
    ev_city = normalize_location(str(event_city)) if pd.notna(event_city) else ''
    ev_state = str(event_state).upper().strip()
    
    # Queue county sometimes contains "County, State" format
    # Check if state appears in queue county
    state_in_queue = ev_state.lower() in q_loc
    if not state_in_queue:
        return 0.0  # Different state = no location match
    
    # Check if city/county name appears in queue county
    if ev_city and ev_city in q_loc:
        return 1.0  # City/County match within same state
    
    # Partial: same state but different city
    return 0.3


def mw_range_match(event_mw, queue_mw, tolerance=0.20):
    """Check if MW values match within tolerance.
    Returns score 0-1."""
    if pd.isna(event_mw) or pd.isna(queue_mw) or event_mw == 0:
        return 0.0
    
    # Both must be positive
    event_mw = float(event_mw)
    queue_mw = float(queue_mw)
    if event_mw <= 0 or queue_mw <= 0:
        return 0.0
    
    ratio = queue_mw / event_mw
    lower = 1.0 - tolerance
    upper = 1.0 + tolerance
    
    if lower <= ratio <= upper:
        # Perfect within range
        return 1.0 - abs(1.0 - ratio)  # Closer to exact = higher score
    elif 0.5 <= ratio <= 2.0:
        # Outside tolerance but within broader range (could be phase 1 vs total)
        return 0.3
    return 0.0


def date_proximity(event_date, queue_date, max_days=180):
    """Check if dates are within max_days of each other.
    Returns score 0-1."""
    if pd.isna(event_date) or pd.isna(queue_date):
        return 0.0
    
    diff = abs((event_date - queue_date).days)
    if diff <= max_days:
        return max(0.1, 1.0 - (diff / max_days))
    elif diff <= max_days * 2:
        return 0.1
    return 0.0


def score_match(event, queue_entry):
    """Calculate composite match score between event and queue entry.
    Weights: company=40, location=30, MW=20, date=10
    Returns score 0-100."""
    
    # Company match (weight: 40)
    company_score = company_name_match(
        event.get('company'),
        queue_entry.get('project_name', '')
    )
    
    # Location match (weight: 30)
    location_score = location_match(
        event.get('location_city'),
        event.get('location_state'),
        queue_entry.get('queue_county')
    )
    
    # MW range match (weight: 20)
    mw_score = mw_range_match(
        event.get('mw_capacity'),
        queue_entry.get('queue_mw')
    )
    
    # Date proximity (weight: 10)
    date_score = date_proximity(
        event.get('announcement_date'),
        queue_entry.get('queue_date')
    )
    
    total = (company_score * 40) + (location_score * 30) + (mw_score * 20) + (date_score * 10)
    return total


print("\u2705 Matching functions defined:")
print("   - company_name_match: normalized name comparison")
print("   - location_match: state + city/county")
print("   - mw_range_match: \u00b120% tolerance")
print("   - date_proximity: within 6 months")
print("   - score_match: composite (company*40 + location*30 + MW*20 + date*10)")

## Step 9: Cross-Reference Events to Queue

For each event, find the best matching queue entry using composite scoring.

In [ ]:
def find_best_queue_match(event, queue_df, min_score=50):
    """Find best queue match for an event.
    Only searches within same ISO region for efficiency."""
    iso = event.get('iso_region')
    
    # Filter queue to same ISO if possible
    if iso and iso in queue_df['iso_region'].values:
        candidates = queue_df[queue_df['iso_region'] == iso]
    else:
        candidates = queue_df
    
    best_score = 0
    best_match = None
    
    for _, qentry in candidates.iterrows():
        score = score_match(event, qentry)
        if score > best_score:
            best_score = score
            best_match = qentry
    
    if best_score >= min_score:
        return best_match, best_score
    return None, best_score


# Process each event
matches = []
unmatched_count = 0
total_events = len(df_events)

print(f"\U0001f50d Cross-referencing {total_events} events against {len(df_queue)} queue projects...")
print(f"   Min match score threshold: 50/100")

for idx, event in df_events.iterrows():
    best_match, score = find_best_queue_match(event, df_queue)
    
    if best_match is not None:
        matches.append({
            'event_idx': idx,
            'match_score': round(score, 1),
            'matched_iso': best_match.get('iso_region'),
            'queue_status': best_match.get('queue_status'),
            'queue_mw': best_match.get('queue_mw'),
            'queue_date': best_match.get('queue_date'),
            'queue_county': best_match.get('queue_county'),
            'actual_completion_date': best_match.get('actual_completion_date'),
            'withdrawn_date': best_match.get('withdrawn_date'),
            'project_name': best_match.get('project_name'),
        })
    else:
        unmatched_count += 1
        matches.append({
            'event_idx': idx,
            'match_score': 0,
            'matched_iso': None,
            'queue_status': None,
            'queue_mw': None,
            'queue_date': None,
            'queue_county': None,
            'actual_completion_date': None,
            'withdrawn_date': None,
            'project_name': None,
        })
    
    if (idx + 1) % 50 == 0:
        print(f"   Processed {idx + 1}/{total_events} events...")

# Convert matches to DataFrame
df_matches = pd.DataFrame(matches)
df_matches = df_matches.set_index('event_idx')

n_matched = (df_matches['match_score'] >= 50).sum()
print(f"\n\U0001f4ca Cross-reference complete:")
print(f"   Matched: {n_matched}/{total_events} (score \u226550)")
print(f"   Unmatched: {unmatched_count}/{total_events}")

## Step 10: Assign Labels

Label each event based on queue status:
- **Operating / Completed** → `kept`
- **Withdrawn / Suspended / Cancelled** → `failed`
- **In Queue / Active / Queued** (no completion date) → `pending`
- **No match found** → `unmatched`

In [ ]:
def assign_label(row):
    """Assign label based on queue status."""
    score = row.get('match_score', 0)
    
    if score < 50:
        return 'unmatched'
    
    status = str(row.get('queue_status', '')).lower().strip()
    
    # Operating / completed = kept
    if any(kw in status for kw in ['operating', 'completed', 'in service', 'commercial operation', 'active']):
        return 'kept'
    
    # Check actual completion date - if it exists and is in the past, it's kept
    acd = row.get('actual_completion_date')
    if pd.notna(acd):
        try:
            if pd.to_datetime(acd) < pd.Timestamp.now():
                return 'kept'
        except:
            pass
    
    # Withdrawn / suspended / cancelled = failed
    if any(kw in status for kw in ['withdrawn', 'suspended', 'cancelled', 'terminated', 'denied', 'rejected']):
        return 'failed'
    
    # Check withdrawn date
    wd = row.get('withdrawn_date')
    if pd.notna(wd):
        try:
            if pd.to_datetime(wd) < pd.Timestamp.now():
                return 'failed'
        except:
            pass
    
    # In queue / active / queued = pending
    if any(kw in status for kw in ['queue', 'active', 'pending', 'queued', 'processing', 'review', 'study']):
        return 'pending'
    
    # Default for matched but ambiguous status
    return 'pending'


# Apply label
df_events['queue_status'] = df_matches['queue_status']
df_events['queue_date'] = df_matches['queue_date']
df_events['queue_mw'] = df_matches['queue_mw']
df_events['queue_county'] = df_matches['queue_county']
df_events['actual_completion_date'] = df_matches['actual_completion_date']
df_events['match_score'] = df_matches['match_score']
df_events['label'] = df_matches.apply(assign_label, axis=1)

# If match_score was already merged, use it
if 'label' not in df_events.columns:
    df_events['label'] = df_events.apply(assign_label, axis=1)

print(f"\U0001f3f7\ufe0f Label distribution:")
label_counts = df_events['label'].value_counts()
for label in ['kept', 'failed', 'pending', 'unmatched']:
    count = label_counts.get(label, 0)
    pct = count / len(df_events) * 100
    print(f"   {label}: {count} ({pct:.1f}%)")

## Step 11: Generate Statistics

Summary by status, by ISO, by company, by year.

In [ ]:
print("\n" + "="*60)
print("\U0001f4ca LABELING SUMMARY")
print("="*60)

# Total
print(f"\n\U0001f4c4 Total events: {len(df_events)}")
print(f"   Matched (score \u226550): {(df_events['match_score'] >= 50).sum()}")
print(f"   Unmatched: {(df_events['match_score'] < 50).sum()}")

# By label
print(f"\n--- By Label ---")
for label, count in df_events['label'].value_counts().items():
    pct = count / len(df_events) * 100
    print(f"   {label}: {count} ({pct:.1f}%)")

# By ISO
print(f"\n--- By ISO Region ---")
iso_label = df_events.groupby(['iso_region', 'label']).size().unstack(fill_value=0)
print(iso_label.to_string())

# By company
print(f"\n--- By Company ---")
company_label = df_events.groupby(['company', 'label']).size().unstack(fill_value=0)
print(company_label.to_string())

# By year (from announcement_date)
print(f"\n--- By Year ---")
year_label = df_events.copy()
year_label['year'] = year_label['announcement_date'].dt.year
year_summary = year_label.groupby(['year', 'label']).size().unstack(fill_value=0)
print(year_summary.to_string())

# Mean match score by label
print(f"\n--- Mean Match Score by Label ---")
print(df_events.groupby('label')['match_score'].mean().round(1).to_string())

## Step 12: Save Output

Save labeled events to `data/processed/buildout_promises_real.csv`.

In [ ]:
# Define output columns
OUTPUT_COLS = [
    'url', 'source_domain', 'announcement_date', 'company',
    'location_city', 'location_state', 'mw_capacity',
    'target_completion_date', 'is_buildout', 'confidence',
    'iso_region', 'queue_status', 'queue_date', 'queue_mw',
    'queue_county', 'actual_completion_date', 'match_score', 'label'
]

# Ensure all output columns exist
for col in OUTPUT_COLS:
    if col not in df_events.columns:
        df_events[col] = None

# Select only output columns for saving
existing_cols = [c for c in OUTPUT_COLS if c in df_events.columns]
df_output = df_events[existing_cols].copy()

# Format dates as strings for CSV
date_cols = ['announcement_date', 'target_completion_date', 'queue_date', 'actual_completion_date']
for col in date_cols:
    if col in df_output.columns:
        df_output[col] = df_output[col].apply(
            lambda x: x.strftime('%Y-%m-%d') if pd.notna(x) else ''
        )

# Ensure output directory exists
os.makedirs('data/processed', exist_ok=True)

# Save
output_path = 'data/processed/buildout_promises_real.csv'
df_output.to_csv(output_path, index=False)
print(f"\U0001f4be Saved {len(df_output)} rows to {output_path}")
print(f"   Columns: {list(df_output.columns)}")
print(f"   File size: {os.path.getsize(output_path):,} bytes")

# DVC tracking
print("\n\U0001f504 Running DVC add...")
try:
    result = subprocess.run(
        ['dvc', 'add', output_path],
        capture_output=True, text=True, check=True
    )
    print(result.stdout)

    # Push to remote
    result_push = subprocess.run(
        ['dvc', 'push', output_path + '.dvc'],
        capture_output=True, text=True
    )
    if result_push.returncode == 0:
        print("\u2705 DVC push successful")
    else:
        print(f"\u26a0\ufe0f DVC push issue (may need remote config): {result_push.stderr}")
except Exception as e:
    print(f"\u26a0\ufe0f DVC step skipped: {e}")
    print("   Run 'dvc add data/processed/buildout_promises_real.csv' manually.")

print(f"\n\u2705 Notebook complete! Output ready for downstream analysis.")

## Summary

✅ **Gridstatus ISO Queue Cross-Reference Complete**

- Loaded buildout events from `data/raw/buildout_events_raw.csv`
- Pulled interconnection queue data from all accessible ISOs (CAISO, MISO, PJM, ERCOT, NYISO, SPP, ISONE)
- Unified queue data into single DataFrame
- Cross-referenced events with queue entries using multi-factor scoring:
  - Company name matching (fuzzy)
  - Location matching (state + county/city)
  - MW range matching (±20%)
  - Date proximity (6 months)
- Labeled each event: kept / failed / pending / unmatched
- Output: `data/processed/buildout_promises_real.csv`
- DVC tracked

**Next steps**:
- Use `data/processed/buildout_promises_real.csv` for ML training and analysis
- Merge with panel data to build predictive model of buildout outcomes